In [1]:
!pip install segmentation-models-pytorch albumentations -q

In [2]:
import os
import cv2
import torch
import numpy as np
from skimage import io
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm
from torchvision import transforms

/home/pathouser1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch
import segmentation_models_pytorch as smp

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate model structure
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,   # no pretrained weights, we’ll load ours
    in_channels=3,
    classes=1,
).to(DEVICE)

# Load fine-tuned weights
model_path = "/home/pathouser1/.cellpose/unet_finetuned_best.pth"  # adjust if saved elsewhere
state_dict = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

print("✅ Fine-tuned model loaded successfully from", model_path)


✅ Fine-tuned model loaded successfully from /home/pathouser1/.cellpose/unet_finetuned_best.pth


In [11]:
import os
import numpy as np
from torch.utils.data import Dataset
from skimage import io
import albumentations as A
from albumentations.pytorch import ToTensorV2


class GlioblastomaDataset(Dataset):
    def __init__(self, image_dir, mask_dir=None, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        self.image_names = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))
        ])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        fname = self.image_names[idx]

        # ---------- Image ----------
        img_path = os.path.join(self.image_dir, fname)
        image = io.imread(img_path)

        # grayscale → RGB
        if image.ndim == 2:
            image = np.stack([image] * 3, axis=-1)

        # RGBA → RGB
        if image.shape[-1] == 4:
            image = image[..., :3]

        # ---------- Mask ----------
        mask = None
        if self.mask_dir is not None:
            mask_name = fname.replace(".png", "_mask.png")
            mask_path = os.path.join(self.mask_dir, mask_name)
            mask = io.imread(mask_path, as_gray=True)

            # binarize + ensure float32
            mask = (mask > 127).astype(np.float32)

        # ---------- Transforms ----------
        if self.transform:
            if mask is not None:
                augmented = self.transform(image=image, mask=mask)
                image = augmented["image"]              # (3, H, W)
                mask = augmented["mask"].unsqueeze(0)   # (1, H, W)
            else:
                augmented = self.transform(image=image)
                image = augmented["image"]

        # ---------- Return ----------
        if mask is not None:
            return image, mask, fname   # 🔑 for validation / metrics
        else:
            return image, fname         # 🔑 for inference only


In [12]:
# =============================================================
# 🧠 1. Imports
# =============================================================
import os, time
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage import io
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# =============================================================
# 🧮 2. Metrics
# =============================================================
def iou_score(pred, gt, eps=1e-7):
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return (inter + eps) / (union + eps)

def dice_score(pred, gt, eps=1e-7):
    inter = np.logical_and(pred, gt).sum()
    return (2 * inter + eps) / (pred.sum() + gt.sum() + eps)

def jaccard_score(pred, gt, eps=1e-7):
    return iou_score(pred, gt, eps)

# =============================================================
# ⚙️ 3. Dataset + Transforms
# =============================================================
val_tfms = A.Compose([
    A.Normalize(),
    ToTensorV2(),
])

img_dir  = "/home/pathouser1/.cellpose/patches"
mask_dir = "/home/pathouser1/.cellpose/gt_masks"
out_dir  = "/home/pathouser1/.cellpose/unet_masks_output"

os.makedirs(out_dir, exist_ok=True)

ds = GlioblastomaDataset(
    image_dir=img_dir,
    mask_dir=mask_dir,
    transform=val_tfms
)

loader = DataLoader(
    ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =============================================================
# ⚙️ 4. Inference + Metrics
# =============================================================
model.eval()
results = []
total_time, count = 0.0, 0

print("🚀 Running U-Net inference + metrics...")

with torch.no_grad():
    for imgs, masks, fnames in tqdm(loader, desc="Inference"):
        start = time.time()

        imgs = imgs.to(DEVICE, non_blocking=True)
        masks = masks.numpy()              # (B,1,H,W)

        preds = torch.sigmoid(model(imgs))
        preds = (preds > 0.5).float().cpu().numpy()

        for i in range(len(fnames)):
            pred = preds[i, 0]
            gt   = masks[i, 0]

            iou  = iou_score(pred, gt)
            dice = dice_score(pred, gt)
            jac  = jaccard_score(pred, gt)

            # save predicted mask
            out_mask = (pred * 255).astype(np.uint8)
            io.imsave(os.path.join(out_dir, fnames[i]), out_mask)

            results.append({
                "filename": fnames[i],
                "IoU": iou,
                "Dice": dice,
                "Jaccard": jac
            })

            count += 1

        total_time += (time.time() - start)

# =============================================================
# 📄 5. Save CSV + Summary
# =============================================================
df = pd.DataFrame(results)
csv_path = os.path.join(out_dir, "inference_metrics.csv")
df.to_csv(csv_path, index=False)

print(f"✅ Stored {count} prediction masks")
print(f"📄 Metrics CSV saved at: {csv_path}")
print(f"📊 Mean IoU: {df.IoU.mean():.4f}")
print(f"📊 Mean Dice: {df.Dice.mean():.4f}")
print(f"⏱ Avg inference time/image: {total_time / count:.3f} sec")


🚀 Running U-Net inference + metrics...


Inference:   0%|          | 1/6733 [00:01<2:40:54,  1.43s/it]/home/pathouser1/.venv/lib/python3.12/site-packages/skimage/_shared/utils.py:328: UserWarning: /home/pathouser1/.cellpose/unet_masks_output/patch_y0_x11776.png is a low contrast image
  return func(*args, **kwargs)
/home/pathouser1/.venv/lib/python3.12/site-packages/skimage/_shared/utils.py:328: UserWarning: /home/pathouser1/.cellpose/unet_masks_output/patch_y0_x12032.png is a low contrast image
  return func(*args, **kwargs)
/home/pathouser1/.venv/lib/python3.12/site-packages/skimage/_shared/utils.py:328: UserWarning: /home/pathouser1/.cellpose/unet_masks_output/patch_y0_x12288.png is a low contrast image
  return func(*args, **kwargs)
/home/pathouser1/.venv/lib/python3.12/site-packages/skimage/_shared/utils.py:328: UserWarning: /home/pathouser1/.cellpose/unet_masks_output/patch_y0_x12544.png is a low contrast image
  return func(*args, **kwargs)
/home/pathouser1/.venv/lib/python3.12/site-packages/skimage/_shared/utils.py:32

✅ Stored 53862 prediction masks
📄 Metrics CSV saved at: /home/pathouser1/.cellpose/unet_masks_output/inference_metrics.csv
📊 Mean IoU: 0.6566
📊 Mean Dice: 0.7746
⏱ Avg inference time/image: 0.005 sec


In [3]:
pip install timm

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import torch
import timm
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# --------------------
# Device
# --------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------
# Paths
# --------------------
img_dir  = "/home/pathouser1/.cellpose/patches"
mask_dir = "/home/pathouser1/.cellpose/unet_masks_output"
csv_path = "/home/pathouser1/.cellpose/xception_patch_mask_features.csv"

# --------------------
# Model
# --------------------
model = timm.create_model(
    "xception",
    pretrained=True,
    num_classes=0,
    global_pool="avg"
)
model.eval().to(device)

# --------------------
# Transforms
# --------------------
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --------------------
# Dataset
# --------------------
class PatchMaskDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.files = [
            f for f in sorted(os.listdir(img_dir))
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
            and os.path.exists(os.path.join(mask_dir, f))
        ]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        img = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, fname)).convert("RGB")

        return self.transform(img), self.transform(mask), fname

# --------------------
# DataLoader
# --------------------
loader = DataLoader(
    PatchMaskDataset(img_dir, mask_dir, transform),
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# --------------------
# CSV init
# --------------------
header_written = os.path.exists(csv_path)

# --------------------
# Inference + Save
# --------------------
with torch.no_grad():
    for batch_idx, (imgs, masks, fnames) in enumerate(
        tqdm(loader, desc="🚀 Extracting Xception features")
    ):
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        img_feats  = model(imgs).cpu().numpy()     # (B, 2048)
        mask_feats = model(masks).cpu().numpy()    # (B, 2048)

        rows = []
        for i in range(len(fnames)):
            concat_feat = np.concatenate([img_feats[i], mask_feats[i]])

            row = {"filename": fnames[i]}
            for j, v in enumerate(concat_feat):
                row[f"f{j}"] = v
            rows.append(row)

        df = pd.DataFrame(rows)

        # Append safely
        df.to_csv(
            csv_path,
            mode="a",
            header=not header_written,
            index=False
        )
        header_written = True

# --------------------
# Done
# --------------------
print("\n✅ Feature extraction finished")
print("📄 CSV saved at:", csv_path)


🚀 Extracting Xception features: 100%|██████████| 1684/1684 [08:32<00:00,  3.29it/s]


✅ Feature extraction finished
📄 CSV saved at: /home/pathouser1/.cellpose/xception_patch_mask_features.csv


In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --------------------
# Paths
# --------------------
feature_csv = "/home/pathouser1/.cellpose/xception_patch_mask_features.csv"
label_csv   = "/home/pathouser1/.cellpose/patch_labels_train.csv"

# --------------------
# Load data
# --------------------
feat_df  = pd.read_csv(feature_csv)
label_df = pd.read_csv(label_csv, header=None)

# Rename label columns safely
label_df.columns = ["filename", "label"]

# --------------------
# Merge on filename
# --------------------
df = pd.merge(label_df, feat_df, on="filename", how="left")
df.to_csv("data.csv",index = False)

print("✅ Matched samples:", len(df))

# --------------------
# Split X / y
# --------------------
X = df.drop(columns=["filename", "label"]).values
y = df["label"].values

# --------------------
# Train-test split (80–20)
# --------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train samples:", X_train.shape[0])
print("Test samples :", X_test.shape[0])

# --------------------
# Feature scaling (VERY IMPORTANT for SVM)
# --------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# --------------------
# SVM model (batch training)
# --------------------
svm = SVC(
    kernel="rbf",        # best default
    C=1.0,
    gamma="scale",
    class_weight="balanced"   # handles imbalance
)

print("🚀 Training SVM...")
svm.fit(X_train, y_train)

# --------------------
# Evaluation
# --------------------
y_pred = svm.predict(X_test)

print("\n📊 Accuracy:", accuracy_score(y_test, y_pred))
print("\n📄 Classification Report:\n", classification_report(y_test, y_pred))
print("\n🧮 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


KeyboardInterrupt: 

In [32]:
import os
import shutil
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --------------------
# Paths
# --------------------
feature_csv = "/home/pathouser1/.cellpose/xception_patch_mask_features.csv"
label_csv   = "/home/pathouser1/.cellpose/patch_labels_train.csv"

patch_img_root = "/home/pathouser1/.cellpose/patches"  # where PNGs actually live
split_root     = "/home/pathouser1/.cellpose/svm_splits_again"

# --------------------
# Load CSVs
# --------------------
feat_df  = pd.read_csv(feature_csv)
feat_df = feat_df.drop_duplicates(subset="filename", keep="first")
label_df = pd.read_csv(label_csv, header=None)
label_df = label_df.iloc[:, :2]
label_df.columns = ["filename", "label"]

# --------------------
# Merge (only matched samples survive)
# --------------------
df = pd.merge(label_df, feat_df, on="filename", how="left")
df.to_csv("data.csv", index=False)

print("✅ Matched samples:", len(df))


✅ Matched samples: 44824


In [44]:
df = df.dropna()

In [33]:
label_df.shape

(44824, 2)

In [34]:
feat_df.shape

(53862, 4097)

In [35]:
feat_df["filename"].value_counts().head()


filename
patch_y9984_x78592.png    1
patch_y0_x0.png           1
patch_y0_x1024.png        1
patch_y0_x10240.png       1
patch_y0_x10496.png       1
Name: count, dtype: int64

In [30]:
feat_df = feat_df.drop_duplicates(subset="filename", keep="first")


In [31]:
feat_df["filename"].value_counts().head()

filename
patch_y9984_x78592.png    1
patch_y0_x0.png           1
patch_y0_x1024.png        1
patch_y0_x10240.png       1
patch_y0_x10496.png       1
Name: count, dtype: int64

In [38]:
tempo = pd.read_csv('data.csv')

/tmp/ipykernel_252820/3476417160.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  tempo = pd.read_csv('data.csv')


In [42]:
tempo = tempo.dropna()

In [43]:
tempo['label'].value_counts()

label
0    42482
1     2214
0      127
Name: count, dtype: int64

In [45]:
df.head()

,filename,label,f0,f1,f2,f3,f4,f5,f6,f7,...,f4086,f4087,f4088,f4089,f4090,f4091,f4092,f4093,f4094,f4095
1,patch_y10240_x14592.png,0,0.288117,0.476953,0.003437,0.011359,0.081940,0.133831,1.486957,0.672174,...,0.892648,0.041434,0.022871,0.0,0.187988,0.131196,0.151164,0.939167,0.798139,0.254650
2,patch_y10240_x14848.png,0,0.781371,0.277937,0.011706,0.000952,0.032080,0.165806,1.666662,0.716602,...,0.156140,0.018137,0.000000,0.0,0.082009,0.238954,0.049598,0.490523,0.771082,0.541409
3,patch_y10240_x15104.png,0,1.065290,0.165279,0.073579,0.004985,0.161587,0.402770,1.272432,0.722132,...,0.000388,0.000000,0.000000,0.0,0.000000,0.058634,0.000000,0.828252,0.259481,0.602379
4,patch_y10240_x15360.png,0,0.765149,0.769290,0.015629,0.001629,0.077563,0.143127,1.417144,0.816109,...,0.263164,0.001124,0.010700,0.0,0.019617,0.047794,0.016551,0.710105,0.576171,0.443480
5,patch_y10240_x15616.png,0,0.625112,0.646031,0.000000,0.014579,0.025271,0.012637,1.568879,0.642987,...,0.773310,0.000000,0.000000,0.0,0.112348,0.011494,0.122312,0.365896,0.818977,0.477967


In [48]:
# --------------------
# Split features / labels
# --------------------
X = df.drop(columns=["filename", "label"]).values
df["label"] = df["label"].astype(int)
y = df["label"].values
filenames = df["filename"].values

# --------------------
# Train-test split
# --------------------
X_train, X_test, y_train, y_test, fn_train, fn_test = train_test_split(
    X,
    y,
    filenames,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train samples:", len(X_train))
print("Test samples :", len(X_test))


Train samples: 35858
Test samples : 8965


In [49]:
# --------------------
# Folder structure
# --------------------
train_0 = os.path.join(split_root, "train", "0")
train_1 = os.path.join(split_root, "train", "1")
test_0  = os.path.join(split_root, "test", "0")
test_1  = os.path.join(split_root, "test", "1")

for d in [train_0, train_1, test_0, test_1]:
    os.makedirs(d, exist_ok=True)

# --------------------
# Copy PNGs
# --------------------
def copy_files(file_list, labels, dst_0, dst_1):
    for fname, lbl in zip(file_list, labels):
        lbl = int(lbl)
        src = os.path.join(patch_img_root, fname)
        if not os.path.exists(src):
            continue
        dst = dst_1 if lbl == 1 else dst_0
        shutil.copy(src, os.path.join(dst, fname))

copy_files(fn_train, y_train, train_0, train_1)
copy_files(fn_test,  y_test,  test_0,  test_1)

print("📂 Train/Test folders created successfully")


📂 Train/Test folders created successfully


In [50]:
# --------------------
# Scaling
# --------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


In [51]:
# --------------------
# SVM model
# --------------------
svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced"
)

print("🚀 Training SVM...")
svm.fit(X_train, y_train)


🚀 Training SVM...


,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [52]:
# --------------------
# Evaluation
# --------------------
y_pred = svm.predict(X_test)

print("\n📊 Accuracy:", accuracy_score(y_test, y_pred))
print("\n📄 Classification Report:\n", classification_report(y_test, y_pred))
print("\n🧮 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



📊 Accuracy: 0.9689905186837702

📄 Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.98      0.98      8522
           1       0.64      0.84      0.73       443

    accuracy                           0.97      8965
   macro avg       0.82      0.91      0.86      8965
weighted avg       0.97      0.97      0.97      8965


🧮 Confusion Matrix:
 [[8313  209]
 [  69  374]]


In [19]:
temp = pd.read_csv('data.csv')

In [20]:
temp.shape

(89646, 4098)

In [21]:
temp1 = pd.read_csv('xception_patch_mask_features.csv')


In [22]:
temp1.shape

(107724, 4097)

In [23]:
temp1.head()

,filename,f0,f1,f2,f3,f4,f5,f6,f7,f8,...,f4086,f4087,f4088,f4089,f4090,f4091,f4092,f4093,f4094,f4095
0,patch_y0_x0.png,0.055026,0.058114,0.000000,0.000762,0.004962,0.026127,1.368782,0.0,0.016209,...,0.000000,0.0,0.000000,0.000000,0.000000,0.106093,0.201086,0.423098,0.000000,1.027103
1,patch_y0_x1024.png,0.026549,0.082159,0.005534,0.000000,0.000000,0.025649,0.950003,0.0,0.029195,...,0.000000,0.0,0.001307,0.123908,0.000000,0.112525,0.000000,0.100181,0.004042,0.781140
2,patch_y0_x10240.png,0.000154,0.269855,0.157960,0.060672,0.194480,0.142432,0.783319,0.0,0.159091,...,0.000000,0.0,0.000000,0.000000,0.000000,0.403872,0.038567,0.200081,0.000000,0.469118
3,patch_y0_x10496.png,0.000000,0.034427,0.257794,0.063071,0.006613,0.033576,0.200986,0.0,0.032009,...,0.002764,0.0,0.079536,0.008805,0.000000,0.338351,0.179363,0.262744,0.011354,0.682847
4,patch_y0_x10752.png,0.000000,0.052376,0.201083,0.041664,0.007318,0.156900,0.252411,0.0,0.007147,...,0.002493,0.0,0.015573,0.216710,0.002685,0.184385,0.149928,0.590630,0.047849,1.270819
